# Stage 31A: TCN ensemble uncertainty shrinkage

Stage 30Aの5 checkpointsを再利用し、ensemble spreadで不確実な補正を縮小します。再学習なし、62 cutsのみです。GPU推奨、予約120 wellsは未使用です。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess,sys
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
import torch
assert torch.cuda.is_available(),'Select a GPU runtime'
print(torch.cuda.get_device_name(0))


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage21b_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
stage30a_run=artifact_dir/'stage30a_residual_tcn_full_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',public_oof_run/'base_oof.parquet',stage21b_run/'confidence_cut_report.parquet',stage30a_run/'summary.json',stage30a_run/'normalizer.npz']+[stage30a_run/f'fold_{fold}.pt' for fold in range(5)]
for path in required: assert path.is_file(),path
stage30=json.loads((stage30a_run/'summary.json').read_text()); assert stage30['stage30a_complete'] and stage30['reserved_confirmation_used'] is False,stage30


In [ ]:
RUN_ID='stage31a_tcn_uncertainty_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve(); assert resolved==expected and resolved.parent==artifact_dir.resolve(); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    command=[sys.executable,'-m','rogii.cli.residual_tcn_uncertainty','--config','configs/experiment/stage31a_tcn_uncertainty.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--stage30a-run',str(stage30a_run),'--validation-run',str(stage21b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID,'--device','cuda']
    audit_env=os.environ.copy(); audit_env['PYTHONPATH']=str(repo_dir/'src')+':'+audit_env.get('PYTHONPATH','')
    result=subprocess.run(command,cwd=repo_dir,env=audit_env,text=True,capture_output=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode: raise RuntimeError(f'Stage 31A failed: {command}')
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage31a_complete','promoted_to_stage31b_reserved_confirmation','device','cuts','wells','ensemble_models','primary_profile','base_rmse','candidate_rmse','rmse_delta','bootstrap_95pct','cut_rmse_p90_delta','gates','reserved_confirmation_used','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['profile_report']).sort_values('rmse_delta')
